# 04 — YOLO26n baseline on PitchLens v002

This notebook trains a first detector on the whole v002 train split, compares it with the pretrained model on the whole validation split, then produces the metrics, matrices, loss curves and visual examples.

The images stay in `data/processed/v002`. Each run only creates a dated folder containing the weights and reports in `models/checkpoints/baseline_runs/`. The test split is never used here.


## 1. Environment and files

Use the project's Python kernel (`.venv`). The dependencies are declared in the `training` extra of pyproject.toml. The cell prints their versions and checks the fingerprint of the downloaded model.


In [ ]:
import csv
import hashlib
import json
import os
import random
import time
from datetime import datetime
from importlib.metadata import version
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.environ["YOLO_CONFIG_DIR"] = str(PROJECT_ROOT / ".ultralytics")
os.environ["YOLO_AUTOINSTALL"] = "false"
os.environ["MPLBACKEND"] = "Agg"

import torch
from IPython.display import Image as DisplayImage
from IPython.display import display
from ultralytics import YOLO, settings

settings.update({"sync": False, "wandb": False, "mlflow": False, "clearml": False})
DATA_ROOT = PROJECT_ROOT / "data/processed/v002"
WEIGHTS = PROJECT_ROOT / "models/checkpoints/yolo26n.pt"
provenance = json.loads((PROJECT_ROOT / "models/reports/yolo26n_pretrained.json").read_text())
assert hashlib.sha256(WEIGHTS.read_bytes()).hexdigest() == provenance["sha256"]
dataset_report = json.loads((DATA_ROOT / "build_report.json").read_text())
assert dataset_report["images"] == 4549
versions = {name: version(name) for name in ("ultralytics", "torch", "torchvision")}
print(versions)
print("MPS available:", torch.backends.mps.is_available())
print("Dataset:", DATA_ROOT)


## 2. Full training and before/after comparison

We use the **whole v002 train split (3,048 images)** and the **whole validation split (670 images)**. We always start from the pretrained `yolo26n.pt` weights.

Local goal: at most 5 epochs with an indicative limit of 10 minutes. Ultralytics checks this limit between certain steps and may therefore finish the current epoch. The Apple GPU is used if available. A real, longer training will be run afterwards on a Hugging Face GPU.

The results cover the validation split only. The test split stays reserved for the final evaluation of the chosen model.


In [ ]:
import sys

import matplotlib.pyplot as plt
from IPython.display import Markdown

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from pitchlens.training.diagnostics import (
    change_category,
    detection_counts,
    quiet_ultralytics_progress,
)
from pitchlens.training.evaluate import BallValidator

torch.set_num_threads(4)
full_parent = PROJECT_ROOT / "models/checkpoints/baseline_runs"
full_parent.mkdir(parents=True, exist_ok=True)
run_timestamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S")
FULL_ROOT = full_parent / f"yolo26n_v002_local_{run_timestamp}"
FULL_ROOT.mkdir()
full_data = DATA_ROOT  # reference v002 directly: no image copy
full_yaml = FULL_ROOT / "dataset.yaml"
full_yaml.write_text(
    "path: " + json.dumps(str(DATA_ROOT)) +
    "\ntrain: train/images\nval: val/images\nnames:\n  0: ball\n"
)
TRAIN_DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
EVAL_DEVICE = "cpu"  # same hardware for both evaluations
evaluation_config = {
    "data": str(full_yaml), "split": "val", "imgsz": 640, "batch": 4,
    "device": EVAL_DEVICE, "workers": 0, "conf": 0.001, "iou": 0.7, "max_det": 300,
    "plots": True, "verbose": False, "project": str(FULL_ROOT),
}
print("Train:", TRAIN_DEVICE, "— comparison:", EVAL_DEVICE)
print("Experiment:", FULL_ROOT)


### 2.1 Before: what can the pretrained model do?

YOLO26n has a COCO class `sports ball`. Our evaluator keeps only these predictions and compares them with the `ball=0` class of v002, negative images included.

The mAP uses all scores from 0.001. The operational figures use **confidence ≥ 0.25 and IoU ≥ 0.50**, identical before and after.


In [ ]:
before_model = YOLO(str(WEIGHTS))
with quiet_ultralytics_progress():
    before_metrics = before_model.val(
        validator=BallValidator, name="before", **evaluation_config
    )
def evaluation_summary(result):
    return {
        "map50": float(result.box.map50),
        "map50_95": float(result.box.map),
        "fixed": result.fixed_threshold,
        "speed_ms_per_image": dict(result.speed),
    }
before_summary = evaluation_summary(before_metrics)
(FULL_ROOT / "before.json").write_text(json.dumps(before_summary, indent=2))
display(before_summary)


### 2.2 Train on v002

The hyperparameters stay simple: AdamW, resolution 640, seed 42 and validation at every epoch. The standard augmentations stay on; mosaic and mixup are disabled for this baseline.

The details go to `training.log`. The notebook only prints one summary per epoch. Images and labels are not copied into the run.


In [ ]:
import gc
import logging

from ultralytics.utils import LOGGER

del before_model
gc.collect()
full_model = YOLO(str(WEIGHTS))
full_config = {
    "data": str(full_yaml), "epochs": 12, "time": 0.75, "imgsz": 640, "patience": 4,
    "batch": 4, "device": TRAIN_DEVICE, "workers": 0, "seed": 42,
    "deterministic": True, "optimizer": "AdamW", "lr0": 0.001,
    "amp": False, "cache": False, "mosaic": 0.0, "mixup": 0.0, "close_mosaic": 0,
    "val": True, "plots": True, "save": True, "verbose": False,
    "project": str(FULL_ROOT), "name": "train", "exist_ok": False,
}
(FULL_ROOT / "experiment.json").write_text(json.dumps({
    "training": full_config, "evaluation": evaluation_config,
    "versions": versions, "pretrained_sha256": provenance["sha256"],
    "dataset_decisions_sha256": dataset_report["decisions_sha256"],
    "comparison_confidence": 0.25, "comparison_iou": 0.5,
    "test_split_used": False,
}, indent=2))
def report_epoch(trainer):
    print(f"Epoch {trainer.epoch + 1} finished — "
          f"validation mAP50: {trainer.metrics.get('metrics/mAP50(B)', 0):.1%}")
full_model.add_callback("on_fit_epoch_end", report_epoch)
old_handlers = list(LOGGER.handlers)
log_handler = logging.FileHandler(FULL_ROOT / "training.log")
LOGGER.handlers = [log_handler]
full_started = time.monotonic()
try:
    with quiet_ultralytics_progress():
        full_model.train(**full_config)
finally:
    LOGGER.handlers = old_handlers
    log_handler.close()
FULL_SAVE_DIR = Path(full_model.trainer.save_dir)
full_active_seconds = time.monotonic() - full_started
print(f"Observed active time: {full_active_seconds/60:.1f} minutes — {FULL_SAVE_DIR}")


In [ ]:
import contextlib

full_best = FULL_SAVE_DIR / "weights/best.pt"
assert full_best.is_file()

# delete full_model only if it exists (avoids a NameError if the training cell
# was not run or the variable has already been deleted)
if "full_model" in globals():
    with contextlib.suppress(NameError):
        del full_model

gc.collect()
if TRAIN_DEVICE == "mps":
    with contextlib.suppress(RuntimeError):
        torch.mps.empty_cache()

after_model = YOLO(str(full_best))
with quiet_ultralytics_progress():
    after_metrics = after_model.val(
        validator=BallValidator, name="after", **evaluation_config
    )
after_summary = evaluation_summary(after_metrics)

with (FULL_SAVE_DIR / "results.csv").open() as stream:
    history = [{k.strip(): float(v) for k, v in row.items()} for row in csv.DictReader(stream)]

comparison = {
    "before": before_summary, "after": after_summary,
    "epochs_completed": len(history),
    "training_seconds_ultralytics": history[-1]["time"],
    "active_seconds_observed": full_active_seconds,
    "best_weights": str(full_best), "validation_images": 670,
    "training_images": 3048, "test_split_used": False,
}
(FULL_ROOT / "comparison.json").write_text(json.dumps(comparison, indent=2))


## 3. Readable summary

- **Precision**: among the detections, what proportion is correct?
- **Recall**: among the annotated balls, what proportion is found?
- **F1**: trade-off between precision and recall.
- **mAP50 / mAP50–95**: quality across several confidence and overlap thresholds.
- **FP / FN**: false detections and missed balls at the fixed threshold.


In [ ]:
rows = [
    ("Precision (conf ≥ 0.25)", before_summary["fixed"]["precision"], after_summary["fixed"]["precision"]),
    ("Recall (conf ≥ 0.25)", before_summary["fixed"]["recall"], after_summary["fixed"]["recall"]),
    ("F1", before_summary["fixed"]["f1"], after_summary["fixed"]["f1"]),
    ("mAP50", before_summary["map50"], after_summary["map50"]),
    ("mAP50–95", before_summary["map50_95"], after_summary["map50_95"]),
]
table = ["| Metric | Before | After | Change |", "|---|---:|---:|---:|"]
for label, before_value, after_value in rows:
    table.append(f"| {label} | {before_value:.1%} | {after_value:.1%} | {(after_value-before_value)*100:+.1f} pts |")
for label, key in (("Correct detections", "tp"), ("False detections", "fp"), ("Missed balls", "fn"), ("False detections on negatives", "fp_on_negatives")):
    a, b = before_summary["fixed"].get(key, 0), after_summary["fixed"].get(key, 0)
    table.append(f"| {label} | {a} | {b} | {b-a:+d} |")
display(Markdown("\n".join(table)))
(FULL_ROOT / "comparison.md").write_text("\n".join(table))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x = list(range(len(rows)))
axes[0].bar([i-0.18 for i in x], [r[1]*100 for r in rows], width=.36, label="Before")
axes[0].bar([i+0.18 for i in x], [r[2]*100 for r in rows], width=.36, label="After")
axes[0].set_xticks(x, ["Precision", "Recall", "F1", "mAP50", "mAP50–95"])
axes[0].set_ylim(0,100)
axes[0].set_ylabel("%")
axes[0].set_title("Same validation, same protocol")
axes[0].legend()
error_rows = [("TP", "tp"), ("FP", "fp"), ("FN", "fn"), ("FP on negatives", "fp_on_negatives")]
error_x = list(range(len(error_rows)))
axes[1].bar([i-0.18 for i in error_x], [before_summary["fixed"].get(k, 0) for _, k in error_rows], width=.36, label="Before")
axes[1].bar([i+0.18 for i in error_x], [after_summary["fixed"].get(k, 0) for _, k in error_rows], width=.36, label="After")
axes[1].set_xticks(error_x, [label for label, _ in error_rows])
axes[1].set_ylabel("Number of boxes")
axes[1].set_title("Detections and errors at the fixed threshold")
axes[1].legend()
fig.tight_layout()
comparison_path = FULL_ROOT / "comparison.png"
fig.savefig(comparison_path, dpi=150)
plt.close(fig)
display(DisplayImage(filename=str(comparison_path)))


## 8. Confusion matrix and learning curves

For a single-class detector, the matrix contains:

- **TP**: ball correctly detected;
- **FP**: predicted box that matches no ball;
- **FN**: annotated ball that was missed;
- the **true background / predicted background** cell is not defined, because an image contains an infinite number of background positions. There is therefore no natural number of true negatives at the box level.

The matrix below uses exactly the same operating point as the table: confidence ≥ 0.25 and IoU ≥ 0.50. The automatic matrices produced by Ultralytics are used for the mAP and rely on a much lower threshold; they must not be compared directly with this table.

The train/validation curves then let us watch for overfitting: if the train loss keeps decreasing while the validation loss rises persistently, the model is probably memorizing the train set. Two epochs only give a trend; more are needed to conclude.

In [ ]:
import numpy as np

# Detection confusion matrix at the fixed operating threshold.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, title, summary in zip(axes, ("Before", "After"), (before_summary, after_summary)):
    fixed = summary["fixed"]
    matrix = np.array([[fixed["tp"], fixed["fp"]], [fixed["fn"], np.nan]])
    masked = np.ma.masked_invalid(matrix)
    axis.imshow(masked, cmap="Blues")
    for row in range(2):
        for column in range(2):
            value = matrix[row, column]
            axis.text(column, row, "N/A" if np.isnan(value) else str(int(value)),
                      ha="center", va="center", fontsize=14,
                      color="white" if not np.isnan(value) and value > np.nanmax(matrix) / 2 else "black")
    axis.set_xticks((0, 1), ("True ball", "True background"))
    axis.set_yticks((0, 1), ("Predicted ball", "Predicted background"))
    axis.set_title(title)
fig.suptitle("Confusion matrix — confidence 0.25 / IoU 0.50")
fig.tight_layout()
confusion_path = FULL_ROOT / "confusion_matrix_fixed.png"
fig.savefig(confusion_path, dpi=150)
plt.close(fig)
display(DisplayImage(filename=str(confusion_path)))

# Learning curves: each train loss is compared with its validation counterpart.
epochs = [row["epoch"] for row in history]
losses = (("box_loss", "Localization"), ("cls_loss", "Classification"), ("l1_loss", "L1"))
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for axis, (key, title) in zip(axes.flat[:3], losses):
    axis.plot(epochs, [row[f"train/{key}"] for row in history], marker="o", label="Train")
    axis.plot(epochs, [row[f"val/{key}"] for row in history], marker="o", label="Validation")
    axis.set_title(f"{title} loss")
    axis.set_xlabel("Epoch")
    axis.set_ylabel("Loss")
    axis.legend()
axes[1, 1].plot(epochs, [row["metrics/mAP50(B)"] * 100 for row in history], marker="o", label="mAP50")
axes[1, 1].plot(epochs, [row["metrics/mAP50-95(B)"] * 100 for row in history], marker="o", label="mAP50–95")
axes[1, 1].set_title("Validation quality")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("%")
axes[1, 1].legend()
fig.suptitle("Learning progress")
fig.tight_layout()
learning_curves_path = FULL_ROOT / "learning_curves.png"
fig.savefig(learning_curves_path, dpi=150)
plt.close(fig)
display(DisplayImage(filename=str(learning_curves_path)))

if len(history) < 4:
    trend_note = ("Only " + str(len(history)) + " epochs: the train and validation losses are decreasing, "
                  "but it is too early to conclude about overfitting.")
else:
    train_down = history[-1]["train/box_loss"] < history[-2]["train/box_loss"]
    val_up = history[-1]["val/box_loss"] > history[-2]["val/box_loss"]
    trend_note = ("Possible warning: train loss decreases but validation loss rises." if train_down and val_up
                  else "No train/validation divergence on the last epoch.")
display(Markdown("**Reading:** " + trend_note))

## 5. Decision curves produced by YOLO

The run folder contains several automatic plots. The four that are useful for choosing the confidence threshold are grouped here:

- **Precision–Recall** summarizes the overall trade-off and its area corresponds to the mAP50;
- **F1–Confidence** shows the threshold giving the best precision/recall trade-off;
- **Precision–Confidence** and **Recall–Confidence** show what is gained and lost as the threshold increases.

The other automatic files remain available as technical artifacts. `results.png` is deliberately omitted because our learning curves are more readable; the Ultralytics matrices use a different threshold, so we present our fixed matrix instead, which is consistent with the table.

In [ ]:
from PIL import Image

curve_files = [
    (FULL_ROOT / "after/BoxPR_curve.png", "Precision–recall"),
    (FULL_ROOT / "after/BoxF1_curve.png", "F1 vs. confidence"),
    (FULL_ROOT / "after/BoxP_curve.png", "Precision vs. confidence"),
    (FULL_ROOT / "after/BoxR_curve.png", "Recall vs. confidence"),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for axis, (curve_path, title) in zip(axes.flat, curve_files):
    axis.imshow(Image.open(curve_path))
    axis.set_title(title)
    axis.axis("off")
fig.suptitle("Choosing the confidence threshold of the trained model")
fig.tight_layout()
decision_curves_path = FULL_ROOT / "decision_curves.png"
fig.savefig(decision_curves_path, dpi=140)
plt.close(fig)
display(DisplayImage(filename=str(decision_curves_path)))

display(Markdown("### What the model actually received during training"))
display(Markdown("This chart describes the positions and sizes of the train boxes. It is mainly useful for spotting a highly imbalanced dataset."))
display(DisplayImage(filename=str(FULL_SAVE_DIR / "labels.jpg"), width=850))

## 6. Before/after visual examples

The gallery uses 32 deterministic images, for qualitative inspection only. The metrics are still computed on the 670 validation images.

The results are split into three short panels:

1. **successful improvements**: an error of the pretrained model has disappeared;
2. **balls still missed**: a green annotation has no matching prediction;
3. **remaining false alarms**: a red box matches no annotated ball.

**Green = reference annotation. Red = model prediction.** A correct overlay can make both colors appear very close together; the titles therefore also describe the outcome in plain language.

In [ ]:
import gc
from collections import Counter

from PIL import Image, ImageDraw

VISUAL_SAMPLE_SIZE = 32
VISUAL_CHUNK_SIZE = 4
VISUAL_BATCH_SIZE = 1

def ground_truth_boxes(image_path):
    with Image.open(image_path) as image:
        image_width, image_height = image.size
    boxes = []
    label_path = full_data / "val/labels" / (image_path.stem + ".txt")
    for line in label_path.read_text().splitlines():
        _, x, y, width, height = map(float, line.split())
        boxes.append(((x-width/2)*image_width, (y-height/2)*image_height,
                      (x+width/2)*image_width, (y+height/2)*image_height))
    return np.asarray(boxes, dtype=float).reshape(-1, 4)

def compact_predictions(model, image_paths, classes=None):
    """Infer small chunks and retain boxes/confidences, never decoded images."""
    compact = []
    with quiet_ultralytics_progress():
        for start in range(0, len(image_paths), VISUAL_CHUNK_SIZE):
            chunk = image_paths[start:start + VISUAL_CHUNK_SIZE]
            results = model.predict(
                [str(path) for path in chunk], classes=classes, conf=.25,
                imgsz=640, batch=VISUAL_BATCH_SIZE, device="cpu", verbose=False,
            )
            compact.extend({
                "boxes": result.boxes.xyxy.cpu().numpy().copy(),
                "confidence": result.boxes.conf.cpu().numpy().copy(),
            } for result in results)
            del results
            gc.collect()
    return compact

all_validation_paths = sorted((full_data / "val/images").glob("*.jpg"))
validation_paths = random.Random(42).sample(all_validation_paths, min(VISUAL_SAMPLE_SIZE, len(all_validation_paths)))
original = YOLO(str(WEIGHTS))
sports_class = next(key for key, value in original.names.items() if value == "sports ball")
before_predictions = compact_predictions(original, validation_paths, [sports_class])
del original
gc.collect()
after_predictions = compact_predictions(after_model, validation_paths)

records = []
for image_path, before_prediction, after_prediction in zip(validation_paths, before_predictions, after_predictions):
    truth = ground_truth_boxes(image_path)
    before_boxes = before_prediction["boxes"]
    after_boxes = after_prediction["boxes"]
    before_counts = detection_counts(truth, before_boxes)
    after_counts = detection_counts(truth, after_boxes)
    records.append({
        "path": image_path, "truth": truth,
        "before_boxes": before_boxes, "before_conf": before_prediction["confidence"],
        "after_boxes": after_boxes, "after_conf": after_prediction["confidence"],
        "before": before_counts, "after": after_counts,
        "category": change_category(before_counts, after_counts),
    })

del before_predictions, after_predictions
category_counts = Counter(record["category"] for record in records)
display(category_counts)
def describe_counts(counts):
    parts = []
    if counts["tp"]:
        parts.append(f"{counts['tp']} ball(s) found")
    if counts["fn"]:
        parts.append(f"{counts['fn']} ball(s) missed")
    if counts["fp"]:
        parts.append(f"{counts['fp']} false alarm(s)")
    return ", ".join(parts) if parts else "no ball and no false alarm"

def choose_examples(predicate, limit=2, excluded=None):
    excluded = excluded or set()
    candidates = [record for record in records if predicate(record) and record["path"].name not in excluded]
    candidates.sort(key=lambda record: (record["after"]["fp"] + record["after"]["fn"], record["path"].name))
    chosen, used_sequences = [], set()
    for record in candidates:
        sequence = record["path"].stem.rsplit("_", 1)[0]
        if sequence not in used_sequences:
            chosen.append(record)
            used_sequences.add(sequence)
        if len(chosen) == limit:
            break
    return chosen

corrected_examples = choose_examples(lambda record: record["category"] == "corrected")
missed_examples = choose_examples(lambda record: record["after"]["fn"] > 0)
false_alarm_examples = choose_examples(
    lambda record: record["after"]["fp"] > 0,
    excluded={record["path"].name for record in missed_examples},
)

def draw_detection(record, moment):
    image = Image.open(record["path"]).convert("RGB")
    drawing = ImageDraw.Draw(image)
    for box in record["truth"]:
        drawing.rectangle(tuple(box), outline="lime", width=5)
    for box, confidence in zip(record[f"{moment}_boxes"], record[f"{moment}_conf"]):
        drawing.rectangle(tuple(box), outline="red", width=3)
        drawing.text((box[0], max(0, box[1]-14)), f"pred. {confidence:.2f}",
                     fill="red", stroke_width=2, stroke_fill="white")
    return image

def save_example_panel(title, examples, filename):
    if not examples:
        display(Markdown(f"**{title}:** no example in the 32-image sample."))
        return None
    fig, axes = plt.subplots(len(examples), 2, figsize=(13, 3.8 * len(examples)), squeeze=False)
    for row, record in enumerate(examples):
        for column, moment in enumerate(("before", "after")):
            axes[row, column].imshow(draw_detection(record, moment))
            label = "BEFORE" if moment == "before" else "AFTER"
            axes[row, column].set_title(f"{label} — {describe_counts(record[moment])}")
            axes[row, column].axis("off")
    fig.suptitle(title, fontsize=16)
    fig.text(.5, .01, "GREEN = annotation   •   RED = prediction", ha="center", fontsize=12)
    fig.tight_layout(rect=(0, .035, 1, .96))
    output_path = FULL_ROOT / filename
    fig.savefig(output_path, dpi=140)
    plt.close(fig)
    display(Markdown(f"### {title}"))
    display(DisplayImage(filename=str(output_path)))
    return output_path

panels = [
    ("Successful improvements", corrected_examples, "examples_corrected.jpg", "corrected"),
    ("Balls still missed", missed_examples, "examples_missed_after.jpg", "missed_after"),
    ("Remaining false alarms", false_alarm_examples, "examples_false_alarms_after.jpg", "false_alarm_after"),
]
for title, examples, filename, _ in panels:
    save_example_panel(title, examples, filename)

error_report = {
    "sample_seed": 42,
    "sample_images": len(validation_paths),
    "population_images": len(all_validation_paths),
    "thresholds": {"confidence": 0.25, "iou": 0.5},
    "sample_category_counts": dict(category_counts),
    "shown_examples": [
        {"image": record["path"].name, "group": group,
         "before": record["before"], "after": record["after"]}
        for _, examples, _, group in panels for record in examples
    ],
}
(FULL_ROOT / "error_analysis.json").write_text(json.dumps(error_report, indent=2))